# LUMIERE patient 048: real anatomy, explicit 2D and 3D growth assumptions

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/arvidl/AI-assisted-glioma-research/blob/main/notebooks/03_lumiere_prepared_slice.ipynb)

Notebook 02 checked a growth model on constructed phantoms, where the right answer is known. This
notebook takes the same model to one real, openly released patient. The question is deliberately
narrow (one MRI plane, two visits four weeks apart, then the same model in 3D). The notebook first runs a deliberately simple
baseline, diagnoses why it fails, and then revises the model along three principles of glioma
biology. It is as interested in *why* the model disagrees with the later scan as in where it agrees.

**Question:** Can a simple migration–proliferation model, started from the automated tumour labels
at one visit, reproduce the automated labels four weeks later, and what does the disagreement
teach us?

**Evidence:** Real prepared MRI and automated-label observations, the expert RANO ratings released
with the dataset, and a conditional numerical model. Retrospective, one patient, one plane: no
expert segmentation, independent forecast or clinical validation. Section 11 needs the full
local volumes and, for its tissue variants, an upstream FreeSurfer parcellation. The only fitted coefficients
(section 9) are labelled as a fit and cannot be tested on a later visit.

**Route through the notebook:** 1 use case and metadata · 2 task, assumptions and tools · 3 data
provenance · 4 observed anatomy · 5–7 a baseline experiment, its controls and sensitivity · 8 a
biologically informed revision · 9 prescribed versus fitted coefficients · 10 an interactive what-if
explorer · 11 the model in 3D, with tissue · 12 time evolution and export · 13 manuscript assets ·
14 results and interpretation · 15 strengths and weaknesses · 16 clinical interpretation and the next study. Read [notebook 02](02_tumor_dynamics.ipynb) first for
the derivation. This notebook needs explicit local prepared data, the released prepared archive or a small slice bundle; it
downloads data only when you run its Colab cell or the explicit `make lumiere-bundle-fetch` / `make lumiere-fetch` steps, and it never runs segmentation, registers scans or substitutes synthetic patient results.

In Google Colab, the first code cell clones the public repository and puts its package on the path; Colab's own NumPy, Matplotlib and nibabel are used, not the locked versions.

## AI contribution and verification

**Question:** see above.

**AI contribution:** three different kinds of AI, kept apart.
- *Trained imaging AI:* the tumour labels come from the MONAI `brats_mri_segmentation` bundle 0.4.8
  (a SegResNet trained on BraTS), run upstream in the investigator's segmentation project on
  registered T1, contrast-enhanced T1 (CT1), T2 and FLAIR. It is not run here. LUMIERE's own
  HD-GLIO-AUTO and DeepBraTumIA segmentations are not used.
- *Generative assistance:* Codex desktop (6 October 2026; model snapshot not captured) helped write
  the loader, solver adapter, controls, sensitivity analysis and figures of the baseline. A Cursor
  agent (12 October 2026; model and settings not captured) wrote this introduction, the metadata
  extraction (`glioma_ai.lumiere_metadata`), the biologically informed revision
  (`glioma_ai.lumiere_biology` and a domain option in the solver), the what-if explorer
  (`glioma_ai.lumiere_interactive`), the 3D extension with an explicit tissue map
  (`glioma_ai.growth_volume`, `glioma_ai.lumiere_volume`), the pinned partial download of
  the released files (`glioma_ai.lumiere_fetch`), the results summary and the strengths and
  weaknesses.
- *Conventional code:* the numerical solver and all figures are ordinary tested code. Rebuilding the
  notebook calls no language model.

**Investigator choices:** the investigator chose LUMIERE Patient 048 and a 2D simplification,
authorized showing the patient's released metadata, and on 12 October 2026 approved the assistant's
proposed revision: a two-level reading, a brain domain, separate scoring of local and distant
lesions, and published coefficients beside a labelled fit. The investigator then asked for
the 3D extension (section 11) and for a reproducible download route.
*Assistant choices awaiting scientific review:* the six visits shown and the week-045→049 pair (to
avoid spanning the resection); selecting the plane from week 045 alone (to prevent leakage from the
later scan); in the baseline, the core labels as seed and target, initial occupancy 0.65,
D = 0.35 mm²/day, ρ = 0.035 day⁻¹ and threshold 0.5; in the revision, the starting levels 0.9
(core) and 0.36 (margin), the CT1 support as brain domain, the local/distant rule, the use of the
Wang et al. (2009) medians and the calibration grid; in 3D, the 25 mm crop, the FreeSurfer-to-tissue tables,
the parenchyma domain, the factor 5 scaled to keep the mean D, and the mean D for lesion voxels
that FreeSurfer did not call grey or white matter; and the wording of the interpretation,
strengths and weaknesses below.

**Verification:**
- Source files are hashed; labels, units and the shared physical grid are checked on load.
- The plane is selected without later data; no-change, diffusion-only and reaction-only controls run
  beside the baseline, and no-change and whole-rectangle controls beside the revision.
- The 3D solver is tested for exact logistic growth, mass conservation in a closed domain,
  symmetry and agreement with the 2D solver; the 3D crop is checked to stay empty at its faces.
- The 36 released Patient-048 files fetched from Figshare by `make lumiere-fetch` were
  byte-identical to the investigator's local copy that the prepared grid was built from.
- Discrete mass balance and time-step halving are checked; the outer boundary stays empty; tests
  check that no flux crosses the brain-domain boundary.
- The released metadata table is extracted by code from the LUMIERE CSV files with their SHA-256
  hashes, and the visit ratings are cross-checked against the visits the loader expects.
- The reading thresholds and coefficients were checked against the cited articles. Wang et al.
  give D in mm²/year in their Methods and Table 2; one results sentence says mm²/day, which would
  imply implausible front speeds.
- The qualitative statements in sections 8, 9, 11 and 14 are asserted in code, so a rerun that
  contradicts them fails.
- *Not verified here:* registration accuracy, segmentation accuracy against expert masks, FreeSurfer's
  tissue classes near the lesion, and any
  biological meaning of the coefficients for this patient.

**Evidence status:**
- *Real-data observations:* MRI planes, automated labels and their areas; released ratings and
  metadata.
- *Synthetic construction:* everything the model produces.
- *Not shown:* a validated growth rate, a forecast, treatment effect or prognosis.

These roles describe development and provenance. [Assistance records](../docs/ai-workflow.md)
retain the review status; record completeness is separate from scientific quality.

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

import numpy as np
from IPython.display import Image, display

try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ModuleNotFoundError:
    IN_COLAB = False


def repository_root():
    candidates = (Path.cwd(), *Path.cwd().parents)
    return next((p for p in candidates if (p / "pyproject.toml").is_file()), None)


if IN_COLAB and repository_root() is None:
    checkout = Path("/content/AI-assisted-glioma-research")
    if not checkout.is_dir():
        url = "https://github.com/arvidl/AI-assisted-glioma-research"
        subprocess.run(["git", "clone", "--depth", "1", url, str(checkout)], check=True)
    pip = [sys.executable, "-m", "pip", "install", "--quiet"]
    subprocess.run([*pip, "nibabel>=5.3,<6"], check=True)
    os.chdir(checkout)
    sys.path.insert(0, str(Path.cwd() / "src"))
ROOT = repository_root()
if ROOT is None:
    raise RuntimeError("Open this notebook in a checkout of the repository or in Google Colab.")

**Google Colab only.** Colab has no local LUMIERE inputs. When you run the next cell there,
it downloads the released prepared Patient-048 inputs (`lumiere_p048_prepared_v1.zip`, 26 MB,
release `lumiere-p048-prepared-v1`), checks the pinned SHA-256, extracts them to `data/` and
verifies every file against the archive's manifest. The files remain under the LUMIERE authors'
non-commercial condition and are not covered by the MIT licence. Outside Colab the cell changes
nothing; there the notebook reads `LUMIERE_P048_ROOT` or `local/lumiere-case.json`
([data guide](../docs/data.md)).

In [ ]:
from glioma_ai.lumiere_dynamics import configured_case_inputs

if not IN_COLAB:
    print("Not Colab: inputs come from LUMIERE_P048_ROOT or local/lumiere-case.json.")
elif configured_case_inputs(ROOT):
    print("Colab: using the configured LUMIERE inputs.")
else:
    from glioma_ai.lumiere_bundle import fetch_bundle

    try:
        fetched = fetch_bundle(ROOT / "data")
    except FileNotFoundError as error:
        print(f"Colab: released inputs not fetched, so sections 3-11 are skipped.\n{error}")
    else:
        os.environ["LUMIERE_P048_ROOT"] = fetched["prepared_root"]
        print(f"Colab: {fetched['files']} files verified; {fetched['use_condition']}.")

## 1. The use case: why one LUMIERE patient?

**Why real anatomy at all.** Phantoms cannot show what goes wrong in practice: irregular tumour
shapes, a resection cavity, treatment, changes of scanner and segmentation errors. Running the
notebook 02 model on one real case shows which of its assumptions survive and which break, which
is what a learner needs before trusting any growth model on patients.

**Why LUMIERE.** [Suter et al. (2022)](https://doi.org/10.1038/s41597-022-01881-7) released
longitudinal MRI of 91 patients with glioblastoma who had pre-operative MRI between 2008 and 2013 at
the Inselspital, Bern, with follow-up until 2017. All were treated with resection and
temozolomide-based chemoradiation. The release contains T1, CT1, T2 and FLAIR, two automated
segmentations, and **expert RANO ratings for each visit** by a neuroradiologist. The images were
skull-stripped and released as NIfTI, and all dates were converted to week counts relative to the
pre-operative scan. Open longitudinal glioblastoma data with expert response ratings are rare.

**Why Patient 048.** The investigator chose this patient, who is also the longitudinal example in the
investigator's published segmentation study
([Lundervold et al. 2026](https://doi.org/10.1080/03091902.2026.2713703)). The released metadata
below show why the case suits teaching:
- all four sequences and both dataset segmentations exist at every one of the six visits;
- the ratings trace a complete course, from complete resection of the enhancing tumour through
  stable disease and complete response to progression, so the case contains a quiet phase and a
  growth phase;
- two progressive-disease visits lie four weeks apart (weeks 045 and 049), and the rating rationale
  records resection only at week 000-2.

The table is generated from `src/glioma_ai/references/lumiere_p048.json`, which
`scripts/extract_lumiere_metadata.py` extracted from the released LUMIERE CSV tables
(ExpertRating v202211, Demographics_Pathology, MRinfo, datacompleteness) with their SHA-256 hashes.

In [ ]:
from IPython.display import Markdown

from glioma_ai.lumiere_metadata import case_reference, scanner_summary

meta = case_reference()
patient_rows = "\n".join(f"| {key} | {value} |" for key, value in meta["patient_level"].items())
display(
    Markdown(
        f"**Table 1. {meta['patient']}: released patient-level metadata.**\n\n"
        "| Field | Released value |\n| --- | --- |\n" + patient_rows
    )
)
visit_rows = "\n".join(
    f"| {v['visit']} | {v['nominal_day']} | {v['rano_rating']} | {v['rating_rationale'] or '–'} | "
    f"{'yes' if v['non_measurable_lesions'] else '–'} | {scanner_summary(v)} | "
    f"{', '.join(v['available'])} |"
    for v in meta["visits"]
)
display(
    Markdown(
        "**Table 2. Visits: expert RANO rating, scanner and available data.** "
        "Nominal day = 7 × week (±3 days; LUMIERE released weeks only).\n\n"
        "| Visit | Nominal day | RANO | Rating rationale | Non-measurable lesions | Scanner | "
        "Available |\n| --- | --- | --- | --- | --- | --- | --- |\n" + visit_rows
    )
)
print("Source:", meta["source"]["article_doi"], "and", meta["source"]["figshare_doi"])
print("Use:", meta["source"]["use_condition"])

**Reading the metadata.**
- *Time.* The released tables count weeks from the pre-operative scan, so week-000-1
  (pre-operative) and week-000-2 (post-operative) share week 0, and Table 2's nominal days come from
  the week labels alone. [Fig. 2 of Suter et al. (2022)](https://www.nature.com/articles/s41597-022-01881-7/figures/2)
  shows an example patient, not named in the caption, whose six visits and ratings match Patient-048
  exactly (down to the five measured target lesions at week 49); it labels the post-operative scan
  **3 days** after the pre-operative one. The code therefore places week-000-2 at day 3
  (`lumiere_slice.VISITS`); that the figure shows this patient is inferred from the match.
- *Course.* Pre-Op → Post-Op with complete resection of the enhancing tumour (CRET) → stable disease
  at week 013 → complete response at week 023 → progressive disease at weeks 045 and 049. At week 045
  the rationale is T2 progression with a new non-measurable lesion; at week 049 it lists T2
  progression, several measured target lesions and a new non-measurable lesion. The progression is
  therefore multifocal, which matters for a model that grows only from existing tumour.
- *Patient.* Table 1 gives sex, age at surgery, IDH and MGMT results and the released overall survival
  time in weeks. IDH status was tested by immunohistochemistry, with sequencing marked as required,
  so it was not confirmed by sequencing. None of these values enters the model.
- *Scanners.* Visits were acquired on two Siemens 1.5 T models; week 045 and week 049, the pair
  compared below, were acquired on different models with different sequence parameters. Part of any
  change between them can therefore be acquisition, not biology.

## 2. The task, the assumptions and the tools

**Task.** Take the week-045 automated labels on one fixed axial plane; turn them into an assumed
tumour-cell density; let that density migrate and grow for 28 nominal days; read the result back as
labels; and compare with the automated labels at week 049, side by side with simpler controls. This
is a **forward "what would this model predict" exercise**, not a forecast, a RANO assessment or a
segmentation evaluation. It is done in three steps:
- a deliberately simple **baseline** (sections 5–7), whose failure is diagnosed;
- a **biologically informed revision** (sections 8–9) with published reading thresholds and
  coefficients, followed by an explicitly labelled fit;
- the revision repeated **in three dimensions** (section 11), with an explicit tissue map.

**Assumptions**, each stated so it can be challenged:

| Layer | Baseline (sections 5–7) | Revision (sections 8–9) | Consequence if wrong |
| --- | --- | --- | --- |
| Registration | Rigid ANTsPyX registration of each visit to the week-000-1 T1, upstream | Same | Cavity collapse, mass effect or misalignment appear as growth or shrinkage |
| Labels | MONAI bundle labels; core = necrotic/non-enhancing (1) + enhancing (4) | Core (1 + 4) read as the T1Gd abnormality; whole union (1 + 2 + 4) as the T2/FLAIR abnormality | Segmentation errors enter both start and target |
| Geometry | One axial plane chosen from week 045 only | Same; section 11: the 3D volume around the lesion | Through-plane growth is invisible or misattributed |
| Initial state | 0.65 inside the core, zero outside | 0.9 in the core, 0.36 in the T2/FLAIR margin, zero outside | Infiltration beyond the T2/FLAIR label is still ignored |
| Domain | Whole image rectangle | Skull-stripped brain support of the week-045 CT1; the cavity is not a barrier. Section 11 also tries FreeSurfer grey and white matter, with CSF closed | Cells spread into background or cavity |
| Biology | Constant isotropic D and logistic ρ; no tissue boundaries, white-matter anisotropy, mass effect or treatment | Same; section 11 also tries $D_\text{white} = 5\,D_\text{grey}$ | No distant lesions; no spread along fibre tracts |
| Coefficients | D 0.35 mm²/day, ρ 0.035 day⁻¹ (assistant's prescription) | Published medians D ≈ 0.081 mm²/day, ρ ≈ 0.041 day⁻¹ (untreated tumours), then a labelled fit | This treated recurrence may differ from the cohort |
| Time | 28 nominal days from week labels | Same | Exact dates and treatment are unknown |
| Reading | Contour where c ≥ 0.5 (uncalibrated) | Core where c ≥ 0.80, T2/FLAIR where c ≥ 0.16 (published convention) | Conventions, not calibrated for these scanners or labels |
| Scoring | Dice against the later core label | Dice against later labels touching the start; distant lesions reported separately | Dice compresses shape into one number |
| Numerics | 1-mm cells, steps ≤ 0.2 day, no flux at the image border | Also no flux at the domain boundary | Checked by balance and step halving |

**Tools.**

| Stage | Tool | Role |
| --- | --- | --- |
| Data | LUMIERE release (Figshare; files labelled CC0, authors' non-commercial condition) | MRI, ratings and metadata |
| Upstream, not rerun here | HD-BET (by the dataset providers) | Skull stripping for anonymization |
| Upstream, not rerun here | ANTsPyX rigid registration (investigator's segmentation project) | All visits on the week-000-1 T1 grid, 1 mm, RAS |
| Upstream, not rerun here | MONAI `brats_mri_segmentation` 0.4.8 (SegResNet) | Automated tumour labels 1, 2 and 4 |
| Upstream, not rerun here | FreeSurfer 8.2.0 on the registered week-045 T1 (investigator's segmentation project) | Tissue parcellation (`wmparc`) for section 11 |
| This notebook | `glioma_ai.lumiere_slice` | Loading, geometry checks, plane selection |
| This notebook | `glioma_ai.lumiere_dynamics` with `glioma_ai.growth_model` | Baseline solver, controls, sensitivity, figures |
| This notebook | `glioma_ai.lumiere_biology` | Two-level reading, brain domain, local/distant scoring, calibration grid |
| This notebook | `glioma_ai.lumiere_interactive`, ipywidgets | What-if explorer (section 10) |
| This notebook | `glioma_ai.growth_volume`, `glioma_ai.lumiere_volume` | 3D solver, tissue tables, 3D local/distant scoring (section 11) |
| Separate explicit step | `glioma_ai.lumiere_fetch` (`make lumiere-fetch`) | Pinned partial download of the released Patient-048 files |
| This notebook | `glioma_ai.lumiere_metadata` | Released metadata with source hashes |
| This notebook | NumPy, nibabel, Matplotlib | Arrays, NIfTI reading, plotting; CPU only |

## 3. Data provenance and explicit preparation

[Suter et al. (2022)](https://doi.org/10.1038/s41597-022-01881-7) released longitudinal glioblastoma MRI, automated segmentations and expert RANO evaluations in LUMIERE. **The labels here are a separate local derivation**, generated upstream with `brats_mri_segmentation` MONAI bundle 0.4.8 using T1c, T1, T2 and FLAIR. They are not expert reference masks or the dataset's original segmentation outputs.

All six selected visits were prepared upstream on the preoperative T1 reference grid. The loader requires matching 3D shapes, coded and consistent affines, RAS directions, orthogonal 1-mm spacing and the recorded label scheme. CT1 must declare millimetres. Prepared label headers may declare unknown units only when the physical affine matches CT1 and the MONAI record specifies 1-mm spacing; this exception is recorded. No registration accuracy or anatomical correctness is inferred from grid agreement.

Configure **one** of:

```sh
export LUMIERE_P048_ROOT=/absolute/path/to/prepared/lumiere_p048
# OR a paired local NPZ/JSON export:
export LUMIERE_P048_SLICE_BUNDLE=/absolute/path/to/lumiere048_slices.npz
```

Alternatively create ignored `local/lumiere-case.json` with `{"prepared_root":"/absolute/path/..."}` or `{"slice_bundle":"/absolute/path/...npz"}`. An optional integer `axial_index` selects a plane explicitly. With no index, the first largest **core** plane in week 045 is selected from that starting segmentation alone. Every other visit uses the same index. Later images never choose the plane.

**Licence and use.** The LUMIERE files are published on Figshare under CC0, but the authors state that the dataset may only be used for non-commercial purposes ([Suter et al. 2022](https://doi.org/10.1038/s41597-022-01881-7), Usage Notes). The MRI slices shown in this notebook and its executed copy are shared for non-commercial teaching and research only and are not covered by this repository's MIT licence. Cite Suter et al. (2022) when reusing them.

In [ ]:
from glioma_ai.lumiere_dynamics import (
    export_slice_case,
    load_configured_case,
    run_case_experiment,
    write_case_paper_assets,
    write_observation_figures,
)

output = ROOT / "outputs/notebook_03"
case = load_configured_case(ROOT)
if case is None:
    print("Real-data example not run: configure an explicit prepared root or paired slice bundle.")
else:
    write_observation_figures(case, output)
    fields = (
        "slice_index",
        "slice_selection",
        "volume_shape",
        "spacing_mm",
        "axis_codes",
        "coordinate_space",
        "label_header_units",
        "label_units_evidence",
        "review",
    )
    print(json.dumps({key: case.metadata[key] for key in fields}, indent=2))
    print("Hashed input files:", len(case.metadata["source_sha256"]))
    display(Image(filename=str(output / "lumiere_observations.png")))

## 4. Describe observed anatomy before modeling it

Label 1 is necrotic/non-enhancing, label 2 edema-associated, and label 4 enhancing tumor. Core is the explicit union $(1,4)$; whole union is $(1,2,4)$. These meanings are retained across visits. **Core includes necrosis, and whole union includes edema: neither is a measured density of viable tumor cells.**

For label $k$, $A_k=\Delta x\Delta y\sum_{ij}\mathbf1\{L_{ij}=k\}$ is a plane area in mm². The stack below shows six fixed-plane observations; its equally spaced visit positions are categorical, not a continuous time axis. It does not interpolate a growth curve across surgery or treatment. Week-000-1 and week-000-2 share the same nominal week bin; no exact perioperative day interval is invented.

The montage uses a separate 1st–99th percentile display window for each CT1. These are display-only choices, recorded in metadata. Intensities are not normalized into comparable biological measurements. Axes show intrinsic in-plane distances along image axes toward R and A; the affine retains their full world-coordinate mapping and slight obliquity.

In [ ]:
if case is not None:
    for row in case.areas():
        print(
            f"{row['visit']:12s} core={row['core_area_mm2']:.1f} mm²; "
            f"edema-associated={row['label_2_area_mm2']:.1f} mm²; "
            f"whole union={row['whole_union_area_mm2']:.1f} mm²"
        )
    display(Image(filename=str(output / "lumiere_areas.png")))

## 5. Specify the forward experiment completely

We compare week 045 with week 049 to avoid spanning the initial resection. The nominal interval is $7(49-45)=28$ days; exact dates and intervening treatment are not modeled. Avoiding the perioperative interval does not imply untreated growth.

| Assumption | Default | Interpretation |
|---|---|---|
| Structural seed $S_0$ | Core labels $(1,4)$ at week 045 | Includes necrotic/non-enhancing tissue; not viable-cell truth |
| Initial state | $c_0=0.65\mathbf1_{S_0}$ | Assumed occupancy, zero outside; no inferred invisible infiltration |
| $D$ | 0.35 mm²/day | Prescribed constant isotropic migration |
| $\rho$ | 0.035 day⁻¹ | Prescribed logistic net growth |
| Duration | 28 nominal days | Derived from visit labels, not exact scan dates |
| Observation | $\hat S_\theta=\{c\ge0.5\}$ | Uncalibrated display rule |
| Domain | Full prepared plane | No flux at outer rectangle; no brain-tissue boundary or out-of-plane transport |
| Numerics | 1-mm cells; steps ≤0.2 day | Exact half reaction, conservative Euler diffusion, exact half reaction |

$$\partial_t c=\nabla\cdot(D\nabla c)+\rho c(1-c),\qquad
M(t)=\int c\,dA,\qquad A_\theta(t)=\int\mathbf1\{c\ge\theta\}\,dA.$$

The MRI slice supplies geometry and a structural seed. It does not supply $c_0$, $D$, $\rho$ or $\theta$. The displayed crop is chosen for viewing only; the model always solves on the full prepared plane. The outer-boundary density is reported so readers can check whether modeled spread reaches that artificial boundary.

In [ ]:
result = None
if case is not None:
    result = run_case_experiment(
        case,
        output,
        selected_labels=(1, 4),
        initial_occupancy=0.65,
        diffusivity_mm2_per_day=0.35,
        rho_per_day=0.035,
        dt_days=0.2,
        threshold=0.5,
    )
    print(json.dumps(result["parameters"], indent=2))
    display(Image(filename=str(output / "lumiere_model.png")))

## 6. Compare against controls, not just against the later scan

The teal contour is the later automated core label; orange is the prescribed model contour. The difference map distinguishes overlap, missed later-label area and additional model area. These discrepancies can reflect model inadequacy, segmentation error, registration error, treatment, tissue deformation or through-plane effects. No single mechanism follows from a mismatch.

Dice is $2|\hat S\cap S_1|/(|\hat S|+|S_1|)$. Area agreement does not imply spatial agreement. Agreement with an automated label is not segmentation accuracy against expert truth and does not assign a RANO category.

The no-change mask is an essential comparator. Pure reaction cannot create support outside the seed; diffusion can reduce a high-threshold area even while a combined model's integrated density increases. Inspect those controls before attributing contour change to proliferation.

In [ ]:
if result is not None:
    for name, values in result["comparisons"].items():
        print(
            f"{name:22s} Dice={values['dice']:.3f}; "
            f"model area={values['predicted_area_mm2']:.1f} mm²; "
            f"missed={values['missed_label_area_mm2']:.1f} mm²; "
            f"additional={values['additional_model_area_mm2']:.1f} mm²"
        )
    display(Image(filename=str(output / "lumiere_controls.png")))

## 7. What changes when we change an assumption?

The grid below varies prescribed $D$ and $\rho$ and repeats the comparison at two uncalibrated thresholds. It is a **retrospective discrepancy landscape**, not a training/validation split, uncertainty distribution or search for a biological patient coefficient. No best-fitting pair is selected here (section 9 selects one, explicitly as a fit). A parameter landscape changes when the observation rule changes.

The record also varies initial occupancy (0.55, 0.65, 1.0) and assumed elapsed time (21, 28, 35 days). These are illustrative timing scenarios, not confidence limits on the actual interval. Their variation should be compared with numerical step-halving error. Numerical convergence alone cannot resolve unknown biology or imaging thresholds.

In [ ]:
if result is not None:
    display(Image(filename=str(output / "lumiere_sensitivity.png")))
    for row in result["initialization_and_timing_sensitivity"]:
        print(
            f"occupancy={row['initial_occupancy']:.2f}, days={row['assumed_days']:2d}: "
            f"area={row['predicted_area_mm2']:.1f} mm², Dice={row['dice']:.3f}"
        )
    print("Numerical checks:", json.dumps(result["numerics"], indent=2))
    assert result["numerics"]["max_discrete_balance_residual_mm2"] < 1e-7
    assert 0 <= result["numerics"]["density_min"] <= result["numerics"]["density_max"] <= 1

## 8. A biologically informed revision

**Why the baseline shrinks.** In sections 5–7 a flat density of 0.65 is read at 0.5. Migration
smooths that step over roughly $\sqrt{2Dt}\approx4$ mm in 28 days, so the edge falls below the
threshold before growth lifts it again: the contour shrinks although the integrated density grows.
The coefficients are not the main problem; the model's front speed $2\sqrt{D\rho}\approx0.22$ mm/day
is comparable with the observed in-plane expansion. The setup is. It also ignores the T2/FLAIR
abnormality, which is what the expert rated as progressing.

**Three principles, three changes.**
1. *MRI shows the tip of the iceberg.* Glioma cells infiltrate beyond the visible abnormality.
   Following [Swanson et al. (2008)](https://doi.org/10.1038/sj.bjc.6604125), the T1Gd boundary
   (here: the core label) is read as about 80% and the T2 boundary (here: the whole union) as about
   16% of the maximum cell density. The start is two-level: 0.9 in the core, 0.36 (between the two
   thresholds) in the T2/FLAIR margin, zero outside. Both levels are assumptions.
2. *Tumour cells move only through brain.* The domain is the support of the week-045 CT1 after skull
   stripping (HD-BET, by the dataset providers) and registration; no flux leaves it. This is the
   dataset's brain mask, not a tissue classification, and the resection cavity is not yet a barrier.
3. *Glioblastoma also recurs at a distance.* Week-049 lesions that do not touch the week-045 labels
   on this plane are scored separately, because no model that grows from the existing tumour can
   produce them.

**Coefficients fixed in advance.** D and ρ are now the published medians of 32 untreated, newly
diagnosed glioblastomas ([Wang et al. 2009](https://doi.org/10.1158/0008-5472.CAN-08-3863)):
D = 29.7 mm²/year ≈ 0.081 mm²/day and ρ = 15/year ≈ 0.041 day⁻¹ (net doubling time 17 days). They
are chosen before looking at week 049, so the comparison is a forward test. This interval follows
treatment, so the cohort is a reference, not this patient's truth.

In [ ]:
from IPython.display import Markdown

from glioma_ai import lumiere_biology as bio

revision = None
if case is not None:
    revision = bio.run_two_level(case)
    split = revision["split"]
    speeds = bio.observed_radial_speeds(case, split)
    variants = bio.compare_variants(case)
    pixel = case.dx_mm * case.dy_mm
    rows = "\n".join(
        f"| {name} | {s['core_area_mm2']:.0f} | {s['whole_area_mm2']:.0f} | "
        f"{s['local_core_dice']:.3f} | {s['local_whole_dice']:.3f} | {s['local_score']:.3f} | "
        f"{s['distant_whole_reached_mm2']:.0f} |"
        for name, s in variants.items()
    )
    display(
        Markdown(
            "**Table 3. Two-level revision, week 045 → 049.** Areas in mm²; Dice against the "
            "week-049 labels that touch the week-045 labels on this plane.\n\n"
            "| Variant | Core reading | T2/FLAIR reading | Local core Dice | "
            "Local T2/FLAIR Dice | Local score | Distant lesion reached |\n"
            "| --- | ---: | ---: | ---: | ---: | ---: | ---: |\n" + rows
        )
    )
    later_whole = split["local_whole"] | split["distant_whole"]
    print(
        f"Week-049 labels on this plane: {split['n_local_components']} local and "
        f"{split['n_distant_components']} distant component(s); distant area "
        f"{split['distant_whole'].sum() * pixel:.0f} mm² "
        f"({split['distant_whole'].sum() / later_whole.sum():.0%} of the later whole union)."
    )
    print(
        f"Observed local expansion of the circle-equivalent radius: core "
        f"{speeds['core_mm_per_day']:.3f} mm/day, T2/FLAIR {speeds['whole_mm_per_day']:.3f} mm/day."
    )
    print(
        f"Published-median front: speed 2√(Dρ) = {revision['front_speed_mm_per_day']:.3f} mm/day, "
        f"margin width √(D/ρ) = {revision['front_width_mm']:.2f} mm."
    )
    print(
        "Labelled cells outside the CT1 brain support, added to the domain: "
        f"{revision['settings']['labelled_cells_added_to_domain']}; balance residual "
        f"{revision['max_balance_residual_mm2']:.1e} mm²."
    )
    figure = bio.write_two_level_figure(case, revision, output, "lumiere_two_level")
    display(Image(filename=str(figure)))
    literature = variants["Literature medians, brain domain"]
    still = variants["No change (two-level reading of week 045)"]
    start_core = case.roi("week-045").sum()
    # The reading below relies on these relations; a rerun that breaks them must fail.
    assert literature["local_whole_dice"] > still["local_whole_dice"]
    assert literature["local_score"] > still["local_score"]
    assert (
        literature["local_score"] > variants["Literature medians, whole rectangle"]["local_score"]
    )
    assert split["local_core"].sum() > 1.5 * start_core
    assert literature["core_area_mm2"] < 0.75 * split["local_core"].sum() * pixel
    assert revision["front_speed_mm_per_day"] < speeds["core_mm_per_day"]
    assert all(v["distant_whole_reached_mm2"] == 0 for v in variants.values())
    assert revision["max_balance_residual_mm2"] < 1e-7

**Reading Table 3.** With published coefficients, the brain domain and the two-level reading, the
model extends the T2/FLAIR margin more like the later labels than no change does, and the local
score improves; confining spread to the brain helps a little. The core reading barely grows, however,
while the local core label grew by more than half. A median untreated glioblastoma advances more
slowly than this tumour did, in line with the dataset authors' description of its course as "a very
fast growth rate". No variant reaches the distant lesion, by construction.

## 9. Prescribed versus fitted coefficients: what one interval can identify

Section 8 fixed D and ρ before looking at week 049. Here we do the opposite and **fit** them: a
grid spanning roughly the published patient ranges (D 0.01–1 mm²/day, ρ 0.008–0.32 day⁻¹) is
searched for the best local score. Week 049 is the last visit, so nothing remains to test the fit
on; its score is in-sample agreement, not a prediction.

The useful question is what the fit can tell us. A Fisher–Kolmogorov front has two scales, its speed
$2\sqrt{D\rho}$ and the width $\sqrt{D/\rho}$ of its invisible margin. The figure marks a line of
equal speed at the observed core expansion (white, dashed) and lines of equal width (yellow).

In [ ]:
if revision is not None:
    grid = bio.calibration_grid(case)
    best = grid["best"]
    spread = bio.identifiability_summary(grid)
    margins = {
        level: bio.calibration_grid(case, margin_level=level)["best"] for level in (0.2, 0.6)
    }
    doubling = np.log(2) / best["rho_per_day"]
    print(
        f"Best grid cell (a fit to week 049): D = {best['diffusivity_mm2_per_day']:.3f} mm²/day, "
        f"ρ = {best['rho_per_day']:.3f} day⁻¹ (net doubling time {doubling:.1f} days)."
    )
    print(
        f"Local score {best['local_score']:.3f} (core Dice {best['local_core_dice']:.3f}, "
        f"T2/FLAIR Dice {best['local_whole_dice']:.3f}); {grid['near_best_cells']} of "
        f"{grid['local_score'].size} cells lie within 0.01 of the best."
    )
    print(
        f"Score range along D at the best ρ: {spread['score_range_along_D']:.3f}; "
        f"along ρ at the best D: {spread['score_range_along_rho']:.3f}."
    )
    for level, fit in margins.items():
        print(
            f"Assumed margin level {level}: best ρ = {fit['rho_per_day']:.3f} day⁻¹, "
            f"best D = {fit['diffusivity_mm2_per_day']:.3f} mm²/day."
        )
    figure = bio.write_identifiability_figure(grid, output, speeds["core_mm_per_day"])
    display(Image(filename=str(figure)))
    assert best["local_score"] > revision["model"]["local_score"]
    assert spread["score_range_along_rho"] > 3 * spread["score_range_along_D"]
    assert margins[0.2]["rho_per_day"] > best["rho_per_day"]
    assert 1 < doubling < 87

**Reading the fit.** The high-score band runs almost vertically: over 28 days the score depends
mainly on ρ and hardly on D across two orders of magnitude. The reason is the two-level start. The
core reading grows mostly because margin cells, assumed at 0.36, grow past 0.80, which takes a time
set by ρ, not because the front travels. The fitted ρ therefore depends on that assumption (lowering
the margin level to 0.2 raises the best ρ; raising it to 0.6 leaves it unchanged on this coarse
grid), and D is essentially not identified by this interval. The fitted doubling
time is shorter than the published median of 17 days but inside the published range of 1–87 days.
A good in-sample score is not evidence of a mechanism: several cells score almost equally, and the
fit absorbs every unmodelled effect (treatment, the scanner change, segmentation and registration
error) into ρ.

## 10. What if? Explore the assumptions yourself

The explorer re-solves the section-8 model (about a quarter of a second per run) whenever a setting
changes; its defaults reproduce section 8 exactly. Four tabs:
- **Spatial:** move to a neighbouring plane (±10 mm; needs the local prepared volumes, not the slice
  bundle), or shift the week-045 label boundaries by ±3 mm to see how much the result depends on the
  segmentation.
- **Temporal:** choose the observation day up to 112 days after week 045, or press play. Only day 28
  has a scan; every other day is a projection of the assumptions, drawn on a shaded background.
- **Tissue:** confine spread to the brain or not, and ask what would happen if migration were faster
  or slower inside the T2/FLAIR-abnormal region. That factor is a what-if, not a known property.
  LUMIERE provides no tissue segmentation; section 11 uses a FreeSurfer map computed upstream,
  and notebook 02's tissue explorer shows the effect on a phantom.
- **Model:** D and ρ on logarithmic sliders, with presets for the published medians, the baseline
  coefficients and the section-9 fit; the starting densities; and the two reading thresholds.
  Invalid combinations, such as a margin density above the core threshold, are refused with a
  message.

**Things to try**
1. With the published medians, play the days. When does the model's core reading reach the week-049
   core area? What does that say about this tumour's pace?
2. Choose the section-9 fit. Why does the T2/FLAIR reading still fall short of the week-049 marker?
   (What does that marker include?)
3. Shift the label boundaries by +2 mm and −2 mm. Is the effect on Dice smaller or larger than
   switching from the published medians to the fit?
4. Lower the T2/FLAIR reading threshold to 0.04. Have you changed an assumption about biology or about
   imaging?
5. Move a few planes up or down. Does the distant lesion stay distant?

The explorer runs only in a live kernel. GitHub shows the static preview below it, for the default
settings at day 28 and day 84.

In [ ]:
from glioma_ai.lumiere_dynamics import configured_case_inputs
from glioma_ai.lumiere_interactive import what_if_explorer

if case is not None:
    prepared = configured_case_inputs(ROOT).get("prepared_root")
    what_if_explorer(case, Path(prepared) if prepared else None)

In [ ]:
import matplotlib.pyplot as plt

from glioma_ai.lumiere_interactive import draw_what_if, what_if_run
from glioma_ai.notebook_display import show_figure

if case is not None:
    preview = what_if_run(case)
    assert preview["at_observed_day"] == revision["model"]
    for day in (28, 84):
        figure = plt.figure(figsize=(14, 5.6), constrained_layout=True)
        draw_what_if(figure, case, preview, day)
        show_figure(figure)

## 11. The same model in three dimensions, with tissue

**Why 3D.** A tumour grows in volume, not in a plane. Sections 5–10 cannot see growth through other
planes, and a lesion that looks distant on one plane might connect to the tumour elsewhere.
Computation is not the obstacle: one 3D run over the cropped volume takes about a second on a
laptop CPU. Data are: 3D needs the full prepared volumes, which stay local under the LUMIERE terms,
not the one-plane slice bundle. Without them this section prints a notice and is skipped.
`make lumiere-fetch` downloads the released Patient-048 files; building the prepared grid from them
is described in [data preparation](../docs/data.md).

**What stays the same.** The equation, the two-level start (0.9 in the core, 0.36 in the T2/FLAIR
margin), the readings (0.80 and 0.16), the published medians and the 28 nominal days.

**What changes.**
- The density lives on the 1 mm grid in 3D, cropped to the week-045 lesion box plus 25 mm. The crop
  uses no later data, and the code checks that the density stays essentially zero at its faces.
- Local and distant week-049 lesions are now 3D face-connected components, and Dice is computed
  against the whole volume.
- A **tissue map**. LUMIERE provides none. We use the FreeSurfer 8.2 parcellation (`wmparc`) that
  the investigator's segmentation project computed on the registered week-045 T1, and group its
  labels into CSF, grey and white matter with explicit tables in `glioma_ai.lumiere_volume`.
  FreeSurfer was built for brains without lesions, so the map is least reliable exactly where it
  matters most, at the tumour.
- Two tissue assumptions are added one at a time:
  - **CSF closed:** cells spread only through grey and white matter and the week-045 lesion, the
    no-flux CSF boundary of [Swanson et al. (2002)](https://doi.org/10.1038/sj.bjc.6600021).
  - **Faster migration in white matter:** $D_\text{white} = 5\,D_\text{grey}$, following
    [Swanson et al. (2000)](https://doi.org/10.1046/j.1365-2184.2000.00177.x). The 2002 paper calls
    the factor 5 a first approximation within an estimated 2–100-fold range. The two values are
    scaled so that the mean D over the domain stays the published median: tissue redistributes
    migration but does not add any. Lesion voxels that FreeSurfer did not call grey or white matter
    get the mean.
- Finally, the section-9 fit, made on one plane, is carried into 3D unchanged.

**Questions to keep in mind.** Does 3D change the verdict of section 8? Does tissue matter over four
weeks? Does a fit made on one plane carry over to the volume?

In [ ]:
from glioma_ai import lumiere_volume as vol
from glioma_ai.lumiere_dynamics import configured_case_inputs

volume_case = volume_runs = None
prepared = configured_case_inputs(ROOT).get("prepared_root")
if revision is not None and not prepared:
    print("3D section not run: it needs the prepared volumes (prepared_root), not a slice bundle.")
elif revision is not None:
    volume_case = vol.load_volume_case(Path(prepared))
    volume_runs = vol.compare_volume_variants(volume_case, fit=best)
    observed = volume_case.metadata["observed"]
    areas = {row["visit"]: row for row in case.areas()}
    plane_factor = (
        areas["week-049"]["whole_union_area_mm2"] / areas["week-045"]["whole_union_area_mm2"]
    )
    volume_factor = observed["later_whole_cm3"] / observed["start_whole_cm3"]
    display(
        Markdown(
            "**Table 4. Observed automated labels in 3D** (cm³; 1 cm³ = 1000 voxels of 1 mm³).\n\n"
            "| Quantity | Week 045 | Week 049 |\n| --- | ---: | ---: |\n"
            f"| Core (labels 1 + 4) | {observed['start_core_cm3']:.1f} | "
            f"{observed['later_core_cm3']:.1f} |\n"
            f"| Whole union (1 + 2 + 4) | {observed['start_whole_cm3']:.1f} | "
            f"{observed['later_whole_cm3']:.1f} |\n"
            f"| Local component (touches week 045): whole / core | | "
            f"{observed['local_whole_cm3']:.1f} / {observed['local_core_cm3']:.1f} |\n"
            f"| Distant components: whole (largest) | | {observed['distant_whole_cm3']:.1f} "
            f"({observed['largest_distant_component_cm3']:.1f}) |\n"
            f"| Week-049 components: local / all | | {observed['n_local_components']} / "
            f"{observed['n_later_components']} |\n\n"
            f"Whole-union growth from week 045 to 049: ×{volume_factor:.1f} in volume, against "
            f"×{plane_factor:.1f} in area on the section-8 plane."
        )
    )
    first = next(iter(volume_runs.values()))
    table = {"No change (two-level reading of week 045)": first["no_change"]}
    table.update({name: run["model"] for name, run in volume_runs.items()})
    unreachable = {name: run["local_outside_domain_fraction"] for name, run in volume_runs.items()}
    rows = "\n".join(
        f"| {name} | {s['core_cm3']:.1f} | {s['whole_cm3']:.1f} | {s['local_core_dice']:.3f} | "
        f"{s['local_whole_dice']:.3f} | {s['local_score']:.3f} | "
        + (f"{unreachable[name]:.0%} |" if name in unreachable else "– |")
        for name, s in table.items()
    )
    display(
        Markdown(
            "**Table 5. The section-8 model in 3D, week 045 → 049.** Volumes in cm³; Dice against "
            "the week-049 3D component that touches the week-045 labels.\n\n"
            "| Variant | Core reading | T2/FLAIR reading | Local core Dice | Local T2/FLAIR Dice | "
            "Local score | Local labels outside the domain |\n"
            "| --- | ---: | ---: | ---: | ---: | ---: | ---: |\n" + rows
        )
    )
    tissue_run = volume_runs.get("Literature, parenchyma, white = 5 × grey")
    if tissue_run is not None:
        tissue, record = tissue_run["tissue"], volume_case.metadata["tissue"]
        fractions = ", ".join(f"{k} {v:.0%}" for k, v in tissue["domain_fractions"].items())
        print(
            f"Tissue map: {record['software']} on {record['input_channel']} (week 045); "
            f"unclassified FreeSurfer labels: {record['unclassified_labels'] or 'none'}."
        )
        print(
            f"Published median spread as D_grey = {tissue['d_grey_mm2_per_day']:.4f} and "
            f"D_white = {tissue['d_white_mm2_per_day']:.4f} mm²/day; domain: {fractions}."
        )
        outside = volume_case.split["local_whole"] & ~tissue_run["domain_mask"]
        csf = int((outside & (volume_case.tissue == vol.CSF)).sum())
        print(
            f"Week-049 local labels outside grey/white matter: {int(outside.sum())} voxels, "
            f"{csf} of them FreeSurfer CSF and {int(outside.sum()) - csf} unclassified."
        )
    print(
        f"Crop {volume_case.metadata['crop_shape']} voxels; largest density on the crop faces "
        f"{max(r['max_density_on_crop_faces'] for r in volume_runs.values()):.1e}; largest "
        f"balance residual {max(r['max_balance_residual_mm3'] for r in volume_runs.values()):.1e} "
        "mm³."
    )
    shown = tissue_run or volume_runs["Literature, brain support"]
    display(Image(filename=str(vol.write_volume_figure(volume_case, shown, output, "lumiere_3d"))))
    curves = vol.write_volume_curves(volume_case, volume_runs, output, "lumiere_3d_curves")
    display(Image(filename=str(curves)))

    literature_3d = volume_runs["Literature, brain support"]["model"]
    fit_3d = volume_runs["2D fit, brain support"]["model"]
    still_3d = first["no_change"]
    plane = case.metadata["slice_index"]
    # The reading below relies on these relations; a rerun that breaks them must fail.
    assert volume_factor > 1.5 * plane_factor
    assert observed["n_local_components"] == 1
    assert observed["largest_distant_component_cm3"] > 15
    assert not volume_case.split["local_full"][:, :, plane][split["distant_whole"]].any()
    assert literature_3d["local_score"] > still_3d["local_score"]
    assert literature_3d["whole_cm3"] < 0.6 * observed["local_whole_cm3"]
    assert fit_3d["local_score"] == max(s["local_score"] for s in table.values())
    assert fit_3d["whole_cm3"] < 0.85 * observed["local_whole_cm3"]
    assert all(r["model"]["distant_reached_cm3"] == 0 for r in volume_runs.values())
    assert all(r["max_density_on_crop_faces"] < 1e-6 for r in volume_runs.values())
    assert all(r["max_balance_residual_mm3"] < 1e-6 for r in volume_runs.values())
    if tissue_run is not None:
        tissue_scores = [
            volume_runs[name]["model"]["local_score"]
            for name in (
                "Literature, parenchyma (CSF closed)",
                "Literature, parenchyma, white = 5 × grey",
            )
        ]
        assert all(abs(s - literature_3d["local_score"]) < 0.02 for s in tissue_scores)
        assert tissue_run["local_outside_domain_fraction"] > 0.1

**Reading Tables 4 and 5.**
- *The volume grew faster than the plane suggests.* The whole union grew more than threefold in
  volume in four weeks, against about twofold in area on the section-8 plane. One plane understates
  this tumour's growth.
- *One connected recurrence plus a genuinely separate lesion.* In 3D, the week-049 labels that touch
  week 045 form a single component. The distant lesion seen on the section-8 plane does not join it
  through any other plane, and the largest separate component is above 20 cm³. "Distant" is
  therefore not an artefact of looking at one plane, which agrees with the new lesion in the
  expert's week-049 rating.
- *The published medians under-predict in 3D as well.* Their T2/FLAIR reading reaches about half of
  the local component's volume. The local score improves on no change, as in section 8.
- *Tissue hardly matters over four weeks, for two reasons.* Closing CSF and weighting white matter
  each change the local score by less than 0.02. First, with the published coefficients the front
  travels only about 3 mm in 28 days ($2\sqrt{D\rho}\cdot 28$ days), and the margin width
  $\sqrt{D/\rho}$ is about 1.4 mm. That reach is too short for the tissue layout to matter much;
  tissue matters for projections over months and for the invisible margin. Second, a fifth of the
  week-049 local labels lie in voxels that the week-045 map calls CSF or leaves unclassified, so no
  model confined to grey and white matter can reach them. Several explanations fit: the growing
  lesion and its oedema may compress the neighbouring sulci (mass effect, which a fixed anatomy
  cannot represent); the automated labels may extend into CSF; or FreeSurfer may misclassify tissue
  near the lesion.
- *A fit made on one plane does not carry over.* The section-9 fit gives the best 3D score but still
  falls well short of the observed volume, because the plane understated the growth. A fit on the
  volume would need faster growth. We do not run it, because the identifiability limits of
  section 9 apply unchanged: one interval mainly constrains ρ.
- In the curves the core reading first *dips*. Migration smooths the 0.9 step below 0.80 before
  growth lifts it again, an artefact of the piecewise-constant start (compare section 8).

## 12. The model in time: an animation for teaching and slides

The section-8 model played day by day from the week-045 scan. Each frame shows, on the left, the
week-045 CT1 plane with the simulated density (colour), its two readings as solid lines (orange:
core, c ≥ 0.80; teal: T2/FLAIR, c ≥ 0.16) and the week-049 automated labels as dashed lines, the
target the model is compared with. On the right, the reading areas so far are drawn against the
labelled areas (markers at weeks 045 and 049). Day 28 is the only day with a scan and is held for a
moment; later days are projections on a grey background. Only the density is simulated; no
intermediate MRI is generated.

**What to watch**
1. The core reading first *shrinks*, to its lowest at day 9, and only regains its starting area
   by day 28: the sharp week-045 edge diffuses below the 0.80 reading before proliferation lifts
   it again. A reading is a threshold on a smooth field, not the tumour itself.
2. The front moves about 0.12 mm/day, roughly 3 mm in four weeks. Compare that with how far the
   dashed week-049 labels lie beyond the week-045 boundary.
3. At day 28 the T2/FLAIR reading falls well short of the labelled whole tumour. More than half of
   the shortfall is the distant lesion, which no amount of spread within this plane reaches.

**Files for slides.** `outputs/notebook_03/lumiere_growth.mp4` (H.264, for PowerPoint and Keynote;
written when an `ffmpeg` executable is available) and `lumiere_growth.gif`, both 1280 × 720 px at
8 frames per second. Every frame's footer carries the source, the LUMIERE non-commercial
condition and the evidence label; keep it when reusing the movie. GitHub cannot play the movie, so
a static strip of days 0, 14, 28 and 56 follows.

**Small reproducible input.** The cell also writes a paired NPZ/JSON export of the six 2D CT1 and
label planes with geometry, display windows, source hashes and a bundle hash; no model weights or
full 3D volumes. It stays in ignored `outputs/`. The same pair can be read after an **explicit
permitted-data upload** by setting `LUMIERE_P048_SLICE_BUNDLE`; do not commit it or upload
unrelated clinical material. The notebook does not perform any upload.

In [ ]:
from glioma_ai.lumiere_movie import movie_strip, write_growth_movie
from glioma_ai.notebook_display import show_figure

if revision is not None:
    movie = write_growth_movie(case, output)
    growth = movie["result"]
    assert growth["at_observed_day"] == revision["model"]
    for kind, path in movie["paths"].items():
        print(f"{kind.upper()}: {path.relative_to(ROOT)} ({path.stat().st_size / 1e6:.1f} MB)")
    width, height = movie["pixels"]
    print(
        f"{movie['frames']} frames, {width} × {height} px, {movie['fps']} frames/s"
        + (f"; MP4 skipped: {movie['mp4_skipped']}" if movie["mp4_skipped"] else "")
    )
    show_figure(movie_strip(case, growth), dpi=80)

    day28 = int(np.flatnonzero(growth["run"].days == 28)[0])
    core, t2 = growth["core_area_mm2"], growth["t2_area_mm2"]
    shortfall = growth["observed"]["later_t2_mm2"] - t2[day28]
    distant = split["distant_whole"].sum() * case.dx_mm * case.dy_mm
    print(
        f"Core reading: {core[0] / 100:.1f} cm² at day 0, lowest {core.min() / 100:.1f} cm² "
        f"(day {growth['run'].days[np.argmin(core)]:.0f}), {core[day28] / 100:.1f} cm² at day 28. "
        f"T2/FLAIR reading at day 28: {t2[day28] / 100:.1f} cm² against "
        f"{growth['observed']['later_t2_mm2'] / 100:.1f} cm² labelled; distant lesion "
        f"{distant / 100:.1f} of the {shortfall / 100:.1f} cm² shortfall."
    )
    # The 'what to watch' notes above rely on these relations.
    assert core.min() < core[0] <= core[day28]
    assert 0.5 * shortfall < distant < shortfall

if case is not None:
    bundle = export_slice_case(case, output / "lumiere048_slices.npz")
    print("Local bundle:", bundle.relative_to(ROOT))
    print("Paired metadata:", bundle.with_suffix(".json").relative_to(ROOT))

## 13. Bring the checked example into the manuscript

Generate local support assets for manuscript review: PDFs and a `.tex` file with code-generated values. The tracked manuscript source is unchanged and does not yet include these Patient-048 panels automatically. Adding them to the paper or talk remains a separate editorial decision after clinical and scientific review.

In [ ]:
if result is not None:
    write_case_paper_assets(result, output, ROOT / "paper/generated")
    print("Local manuscript panels and generated numbers are ready for make paper.")

## 14. Results and interpretation

**What was observed.** On this plane the automated core and the whole tumour union both grew
markedly between week 045 and week 049 (Table R1), and part of the later label is a separate lesion
that does not touch the week-045 labels. This agrees with the expert's rating of progressive disease
with T2 progression and a new lesion at week 049, although the rating uses all planes and its own
measurements, not these labels.

**The baseline (sections 5–7).** With the prescribed coefficients the thresholded area ends *below*
its starting area, and its overlap with the later label is no better than assuming nothing changed.
Reaction alone reproduces the no-change result exactly, because logistic growth cannot add area
outside the seed. Tuning can match the later *size* but not its *shape*. Section 8 shows why: the
flat start read at one threshold, not mainly the coefficients.

**The revision (section 8).** With a two-level reading, a brain domain and published coefficients
fixed in advance, the model improves on no change for the tumour that was already there, mainly in
the T2/FLAIR margin. Its core grows too slowly: this tumour progressed faster than a median untreated
glioblastoma.

**In three dimensions (section 11).** The whole union grew more than threefold in volume, much
more than on the plane. The published medians again improve on no change but reach about half
the observed local volume. Closing CSF and faster white-matter migration hardly change
four-week results, partly because a fifth of the later local labels lie in voxels the
week-045 map calls CSF or leaves unclassified. The plane fit carried into 3D still
under-predicts the volume.

**The fit (section 9).** Tuned to week 049, the local agreement rises further, but the interval
identifies essentially one quantity, ρ, and only given the assumed margin density; D stays
unidentified. With no later visit, the fit cannot be tested.

**How to interpret the remaining disagreement.** Several explanations fit, and one case cannot
separate them:
- *Distant progression:* a separate lesion on this plane, and new lesions recorded in the ratings at
  weeks 045 and 049; a model that grows from the existing tumour cannot create them.
- *Infiltration beyond the T2/FLAIR label:* cells outside every starting label, which the model
  sets to zero, could become visible by week 049.
- *Through-plane growth:* tumour may enter this plane from neighbouring slices. Section 11
  confirms that the volume grew faster than the plane, and that the distant lesion is separate
  in 3D too.
- *Measurement:* the two visits come from different scanner models; automated labels vary between
  scans; rigid registration leaves residual misalignment.
- *Treatment and biology* that the model omits, such as mass effect. White-matter weighting
  barely changes four-week results (section 11).

**What is not concluded.** No validated growth rate or migration coefficient is claimed for this
patient (the fitted ρ is conditional and untested); nothing is forecast; no treatment effect,
response category or prognosis is inferred. The value of the exercise is that each assumption is
visible, so each disagreement points to a specific next question.

In [ ]:
if result is not None:
    areas = {row["visit"]: row for row in case.areas()}
    start, later = (areas[v]["core_area_mm2"] for v in ("week-045", "week-049"))
    whole_start, whole_later = (areas[v]["whole_union_area_mm2"] for v in ("week-045", "week-049"))
    comparisons = result["comparisons"]
    model, still = comparisons["Reaction + diffusion"], comparisons["No change"]
    grid_rows = result["coefficient_sensitivity"]
    dice = [row["dice"] for row in grid_rows]
    grid_areas = [row["predicted_area_mm2"] for row in grid_rows]
    missed = [row["missed_label_area_mm2"] for row in grid_rows]
    literature = variants["Literature medians, brain domain"]
    unchanged = variants["No change (two-level reading of week 045)"]
    rows = [
        (
            "Core area, week 045 → 049 (mm²)",
            f"{start:.0f} → {later:.0f} ({later / start - 1:+.0%})",
        ),
        (
            "Whole union, week 045 → 049 (mm²)",
            f"{whole_start:.0f} → {whole_later:.0f} ({whole_later / whole_start - 1:+.0%})",
        ),
        (
            "Distant lesion on this plane (mm², share of later whole union)",
            f"{split['distant_whole'].sum() * pixel:.0f} "
            f"({split['distant_whole'].sum() / later_whole.sum():.0%})",
        ),
        (
            "Baseline: default model area (mm²), core Dice; no change Dice",
            f"{model['predicted_area_mm2']:.0f}, {model['dice']:.3f}; {still['dice']:.3f}",
        ),
        (
            f"Baseline sensitivity ({len(grid_rows)} cells): area (mm²), Dice",
            f"{min(grid_areas):.0f}–{max(grid_areas):.0f}, {min(dice):.3f}–{max(dice):.3f}",
        ),
        (
            "Revision, published medians: local core / T2/FLAIR Dice",
            f"{literature['local_core_dice']:.3f} / {literature['local_whole_dice']:.3f}",
        ),
        (
            "Revision, no change: local core / T2/FLAIR Dice",
            f"{unchanged['local_core_dice']:.3f} / {unchanged['local_whole_dice']:.3f}",
        ),
        (
            "Fit to week 049 (in-sample): local score; D (mm²/day), ρ (day⁻¹)",
            f"{best['local_score']:.3f}; {best['diffusivity_mm2_per_day']:.3f}, "
            f"{best['rho_per_day']:.3f}",
        ),
    ]
    display(
        Markdown(
            "**Table R1. Results on the fixed plane (automated labels).**\n\n"
            "| Quantity | Value |\n| --- | --- |\n"
            + "\n".join(f"| {name} | {value} |" for name, value in rows)
        )
    )
    # Section 14's interpretation relies on these relations; a rerun that breaks them must fail.
    assert later > 1.3 * start and whole_later > 1.5 * whole_start
    assert model["predicted_area_mm2"] < start < later
    assert model["dice"] <= still["dice"]
    assert comparisons["Reaction only"]["dice"] == still["dice"]
    assert max(grid_areas) > later and max(dice) < 0.8
    assert min(missed) > 0.15 * later
    assert split["distant_whole"].sum() > 0.15 * later_whole.sum()

## 15. Strengths and weaknesses

**Of the approach**

| Strengths | Weaknesses |
| --- | --- |
| Every assumption is explicit, and the important ones are varied | Sections 5–10 use one plane; section 11 adds 3D, still for one interval and a fixed anatomy |
| A failing baseline is diagnosed and revised for stated biological reasons, which is how models should be improved | Isotropic migration; section 11's grey/white weighting is a fixed 5:1 assumption on a FreeSurfer map that is least reliable near the lesion; no fibre anisotropy, mass effect or treatment |
| Controls: no change, diffusion only and reaction only in the baseline; no change and whole rectangle in the revision | Piecewise-constant starting densities are assumptions; the 80%/16% reading is a convention for T1Gd/T2 images, transferred here to automated labels |
| Reading thresholds and coefficients come from published sources and are fixed before the comparison; the fit is labelled as one | The published coefficients describe untreated, pre-operative tumours; this interval follows treatment |
| The fit is used to show what one interval can and cannot identify | One interval identifies at most ρ, conditional on the margin assumption; the fit cannot be tested on a later visit |
| The plane is chosen without later data; exact checks of mass balance, step halving and the domain boundary | Automated labels, not expert masks; rigid registration only; nominal week-based time |
| Hashed inputs, a small re-runnable slice bundle, CPU in seconds, the tested solver from notebook 02 | Dice compresses overlap into one number and is sensitive to small or scattered structures |
| 3D with an explicit, hashed tissue map; the released files can be fetched reproducibly by pinned hashes | Registration, labels and tissue maps are local derivatives that cannot be downloaded; 3D needs the full prepared volumes |

**Of the use case**

| Strengths | Weaknesses |
| --- | --- |
| Openly released, anonymized data with expert RANO ratings per visit | One patient: nothing generalizes, and the case was chosen rather than sampled |
| Complete four-sequence protocol and both dataset segmentations at all six visits | Only two visits feed the model; the other four are context |
| A full clinical course, including a quiet phase and progression | The compared interval follows treatment, so it is not untreated growth |
| Cross-reference to the investigator's published segmentation of the same case | Distant progression is beyond a model that grows from the existing tumour |
| Small enough to inspect completely while teaching | Scanner models differ between the compared visits; dates are released only as weeks; reuse is limited by the authors' non-commercial condition |

## 16. Clinical interpretation and the next AI investigation

This example links automated imaging to a transparent biological modeling question. A mismatch is useful for generating hypotheses, but two treated visits, on one plane or in 3D, cannot identify a pure growth rate, treatment effect, cell-level mechanism, cognition outcome or recurrence prediction. The six-visit montage is contextual; it is not six untreated observations fitted by this growth model.

A future AI study could compare a conventional model, a regularized predictor and an imaging-informed model on **patient-separated chronological test data**. Define the target first (future compartment area/shape or separately measured cognition), use only information available at baseline, retain a no-change comparator, and assess calibration and error under segmentation/registration variation. Slice selection and parameter fitting must occur without later test scans. More independent patients and exact timing/treatment information are required. Neither that predictive study nor a productivity benefit evaluation is performed here.

**Exercise:** rerun in a different output directory with enhancing label $(4,)$ or whole union $(1,2,4)$. Keep the same fixed plane. Explain what biological state each choice would conflate, rather than selecting whichever union gives the largest Dice. The brain domain of section 8 is the support of the skull-stripped image, not a tissue classification. Section 11's tissue map is an unvalidated FreeSurfer parcellation of a brain with a lesion. Trusting tissue-dependent migration needs a validated cavity and tissue mask and a model of mass effect; CT1 thresholding alone does not supply them.

**Outputs:** six-visit MRI montage, compartment-area table, model snapshots/player, controls, threshold/coefficient sensitivity, the two-level revision and the identifiability map, 3D orthogonal views and volume curves, initialization/timing scenarios, numerical checks and input/source/environment/output hashes. All acquired data and generated artifacts remain ignored. See [notebook 02](02_tumor_dynamics.ipynb) for the mathematics and [data preparation](../docs/data.md).